In [1]:
# Mount Google Drive to access dataset files stored in Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# =========================
# LOAD DATA
# =========================
import pandas as pd
import numpy as np
import re

df = pd.read_csv("/content/drive/My Drive/EADA/EADA_DeepLearning/Project/merged_dataset.csv")

START_IDX = 12499

/tmp/ipykernel_5503/2386184368.py:8: DtypeWarning: Columns (0,4,5,11,12,13,17,18,19,21,23,24,25,26,27,30,31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/drive/My Drive/EADA/EADA_DeepLearning/Project/merged_dataset.csv")


In [3]:
# =========================
# HELPER FUNCTIONS
# =========================

def normalize_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

def ensure_list(x):
    if isinstance(x, list):
        return x
    elif pd.isna(x) or x is None:
        return []
    else:
        return [x]

In [4]:


# =========================
# 1. BASIC FIELD FIXES
# =========================

df.loc[START_IDX:, 'authors'] = df.loc[START_IDX:, 'authors'].fillna('Unknown')
df.loc[START_IDX:, 'description'] = df.loc[START_IDX:, 'description'].fillna('')
df.loc[START_IDX:, 'page_count'] = df.loc[START_IDX:, 'page_count'].fillna(0)

# Normalize text
for col in ['title','subtitle','authors','categories']:
    df.loc[START_IDX:, col] = df.loc[START_IDX:, col].astype(str).str.strip().str.lower()


In [5]:

# =========================
# 2. TITLE CLEAN
# =========================

df.loc[START_IDX:, 'title_clean'] = df.loc[START_IDX:, 'title'].str.replace(
    r'[^a-z0-9]', '', regex=True
)



In [6]:

# =========================
# 3. READING TIME + LENGTH
# =========================

df.loc[START_IDX:, 'reading_time'] = df.loc[START_IDX:, 'page_count'] / 30

df.loc[START_IDX:, 'length_category'] = pd.cut(
    df.loc[START_IDX:, 'page_count'],
    bins=[0,150,400,1000],
    labels=['short','medium','long']
)



In [7]:

# =========================
# 4. CATEGORY FALLBACK (LIGHT VERSION)
# =========================

def infer_from_title(title):
    t = str(title).lower()
    if 'guide' in t or 'how to' in t:
        return ['self-help']
    if 'history' in t:
        return ['history']
    if 'science' in t:
        return ['science']
    return []

df.loc[START_IDX:, 'categories'] = df.loc[START_IDX:].apply(
    lambda row: infer_from_title(row['title']) if pd.isna(row['categories']) or row['categories'] == '' else row['categories'],
    axis=1
)

df.loc[START_IDX:, 'category_source'] = 'filled_lite'



In [8]:

# =========================
# 5. COMBINED TEXT
# =========================

df.loc[START_IDX:, 'authors'] = df.loc[START_IDX:, 'authors'].apply(
    lambda x: x if isinstance(x, list) else [x]
)

df.loc[START_IDX:, 'categories'] = df.loc[START_IDX:, 'categories'].apply(ensure_list)

df.loc[START_IDX:, 'combined_text'] = (
    df.loc[START_IDX:, 'title'] + ' ' +
    df.loc[START_IDX:, 'subtitle'].fillna('') + ' ' +
    df.loc[START_IDX:, 'authors'].apply(lambda x: ' '.join(x)) + ' ' +
    df.loc[START_IDX:, 'categories'].apply(lambda x: ' '.join(x)) + ' ' +
    df.loc[START_IDX:, 'description']
)



In [9]:

# =========================
# 6. READING SPEED VARIANTS
# =========================

df.loc[START_IDX:, 'reading_time_fast'] = df.loc[START_IDX:, 'page_count'] / 30
df.loc[START_IDX:, 'reading_time_slow'] = df.loc[START_IDX:, 'page_count'] / 20



In [10]:

# =========================
# 7. MOOD DETECTION
# =========================

mood_keywords = {
    'dark': ['death','war','murder','crime'],
    'inspiring': ['success','growth','achievement'],
    'romantic': ['love','relationship','heart'],
    'adventurous': ['journey','quest','explore'],
    'educational': ['guide','learn','study'],
}

def detect_mood(text):
    text = str(text).lower()
    scores = {m: sum(w in text for w in words) for m, words in mood_keywords.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'neutral'

df.loc[START_IDX:, 'mood'] = df.loc[START_IDX:, 'combined_text'].apply(detect_mood)



In [12]:
# =========================
# FIX MISSING published_date
# =========================

def fix_published_date(x):
    try:
        return pd.to_datetime(x, errors='coerce')
    except:
        return pd.NaT

df.loc[START_IDX:, 'published_date'] = df.loc[START_IDX:, 'published_date'].apply(fix_published_date)


# =========================
# FIX authors STRUCTURE
# =========================

df.loc[START_IDX:, 'authors'] = df.loc[START_IDX:, 'authors'].apply(
    lambda x: [x] if isinstance(x, str) else ([] if pd.isna(x) else x)
)


# =========================
# FIX page_count VALIDITY
# =========================

df.loc[START_IDX:, 'page_count'] = df.loc[START_IDX:, 'page_count'].apply(
    lambda x: x if pd.notna(x) and x > 0 else 100  # fallback default
)


# =========================
# FIX length_category (strict)
# =========================

def fix_length(row):
    pc = row['page_count']
    if pc <= 150:
        return 'short'
    elif pc <= 400:
        return 'medium'
    else:
        return 'long'

df.loc[START_IDX:, 'length_category'] = df.loc[START_IDX:].apply(fix_length, axis=1)

In [13]:

# =========================
# 8. SAVE FIXED DATASET
# =========================

output_path = "/content/drive/My Drive/EADA/EADA_DeepLearning/Project/merged_filled.csv"
df.to_csv(output_path, index=False)

print("Fix complete. Saved to:", output_path)

Fix complete. Saved to: /content/drive/My Drive/EADA/EADA_DeepLearning/Project/merged_filled.csv
